# M4 · Faster computation — attention backend & INT8 weights

**Learning objective:** validate two *faster-computation* levers on your W7900 at the prescribed 40-step workload: F1 = ck attention kernel, F2 = INT8-quantized DiT weights. Both arms change execution speed, not the amount of sampling — so quality gates use **fidelity semantics**. A measured regression is a valid scientific outcome and stays visible.

**Predict first:** F1 ≈ 45.1 s and F2 ≈ 51.1 s medians on the reference host — but attention backend effects are architecture-specific; write your own expectation.

Reference lesson: [M04 lesson](../course/M04_faster_computation/LESSON.md) · [中文](./04_faster_computation.zh-CN.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse the latest physical run (shown run_id)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="04")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

def want_own_arms():
    """Whether this session executes its own bounded arms (True unless the
    student opted to reuse the latest physical run to save GPU time)."""
    if session.mode == "REFERENCE" or RUN_OWN_GPU_ARMS:
        return True
    print("RUN_OWN_GPU_ARMS=False: reusing the latest physical campaign run(s) "
          "instead of launching new ones (metered-GPU mode)")
    return False

## 1 · Run F1 (ck attention) and F2 (INT8) on YOUR W7900
B0 comes from M1/this campaign (shown run_id). Each arm boots a fresh server with its pinned flags — the runner refuses to benchmark when the attention backend or model silently falls back.

In [ ]:
def run_faster_arms():
    prof = "classroom" if session.mode == "GPU_CLASSROOM" else "research"
    b0 = session.run("B0", profile=prof)
    f1 = session.run("F1", profile=prof)
    f2 = session.run("F2", profile=prof)
    return b0, f1, f2

arms = gpu_cell(run_faster_arms) if want_own_arms() else None
if arms is None:
    ids = [session.latest_campaign_run(x) for x in ("B0", "F1", "F2")]
    if all(ids):
        arms = tuple(session.consume(i, why=f"campaign {x}")
                     for i, x in zip(ids, ("B0", "F1", "F2")))
if arms is not None:
    ab_f1 = rt.ab_compare(arms[0], arms[1], label="B0 vs F1 (ck attention)")
    ab_f2 = rt.ab_compare(arms[0], arms[2], label="B0 vs F2 (int8 weights)")
    for ab in (ab_f1, ab_f2):
        print({k: ab[k] for k in ("classification", "speedup_median",
                                  "latency_delta_s", "vram_delta_mb")})

## 2 · Verify the optimization was actually active
A claimed attention win means nothing if the server silently ran the default backend. The records carry scraped server-log evidence — assert it before interpreting speedups.

In [ ]:
def attention_evidence(bundle):
    evs = [r.get("attention_evidence") or {} for r in bundle.records]
    active = [e for e in evs if e]
    if not active:
        return "NO_EVIDENCE_IN_RECORDS"
    return active[-1]

if arms is not None:
    for name, b in (("F1", arms[1]), ("F2", arms[2])):
        st = b.stats()
        print(f"{name}: attn={st['attention_backend']} unet={st['unet']} "
              f"precision={st['precision']}")
        print(f"   server-log evidence: {attention_evidence(b)}")
    b0s = arms[0].stats()
    print("B0 attention:", b0s["attention_backend"],
          "| F1 must show the ck backend ACTIVE, F2 the int8 unet file")

## 3 · Read the numbers honestly
1. Never multiply independent speedups: measured F1×F2 ≠ a measured combined effect (interactions are real; that is M6's job).
2. INT8 often pays in **VRAM** more than in raw speed — quantify both.
3. If your measured F1 is a regression, report it as measured.

In [ ]:
if arms is not None:
    b0s, f1s, f2s = (a.stats() for a in arms)
    print(f"{'arm':4s} {'median_s':>9s} {'n':>3s} {'vram_dev_mb':>11s}")
    for nm, st in (("B0", b0s), ("F1", f1s), ("F2", f2s)):
        print(f"{nm:4s} {st['median_s']:9.2f} {st['n_warm']:3d} "
              f"{st['vram_device_peak_mb'] or st['vram_peak_mb']:11.0f}")
    print("\nINT8 VRAM saving (MB):",
          (b0s['vram_device_peak_mb'] or b0s['vram_peak_mb'])
          - (f2s['vram_device_peak_mb'] or f2s['vram_peak_mb']))
    for ab, nm in ((ab_f1, "F1"), (ab_f2, "F2")):
        _stamp(MEASURED_TODAY,
               f"{nm} speedup {ab['speedup_median']}x [{ab['classification']}]")
    print("reminder: do NOT report F1*F2 as a combined measured gain")

## 4 · Fidelity semantics for the quality gate
F1/F2 keep the same 40-step schedule as B0, so paired images are checked as *numerics variants* (LPIPS fidelity threshold triages human review; it never auto-passes).

> **Quality decisions stay human.** No cell auto-approves an image-quality verdict. Historical owner verdicts apply **only** to their originally evaluated deployment scope; a new physical run needs its own review (`PENDING_HUMAN_REVIEW` until then).

In [ ]:
print("run your own fidelity gate in GPU modes:")
print("  session.run_quality('F1', reference='B0')")
print("  session.run_quality('F2', reference='B0')")
print("frozen campaign artifacts: evidence/quality/F1, evidence/quality/F2")
_stamp(HISTORICAL, "reference F2 fidelity gate triaged human review; INT8 "
      "numerical FAIL (in M5.1's H3_OV_INT8 encoder) stays FAIL - never promoted")

## 5 · Your turn
1. Explain why ck attention might help one ROCm arch and not another.
2. INT8 changed VRAM by ~6.6 GB on the reference host — what does that enable on a 48 GB card?
3. Design the one-arm experiment that would isolate *dequantization overhead* in F2.

## Acceptance criteria

- [ ] You can state which numbers are **measured on this GPU today**, which are
      **historical reference values**, and which are **estimates/simulations**.
- [ ] Every physical run you produced has a visible `run_id` under `runs/`.
- [ ] You did not present a smoke/classroom population as a formal result.
- [ ] You can explain one limitation of each measurement you reported.

## Reproduce & contribute

```bash
python -m lab doctor                                  # admission evidence -> runs/doctor.json
python -m lab run --experiment F1 --date <your-tag>  # full research protocol
python -m lab report --scope student --kind all       # aggregate YOUR runs only
```

Commit your `runs/<run-id>/*.json` records (never `evidence/`) and open a PR
describing: hypothesis → protocol → numbers → limitations. Negative results
are welcome — a measured regression is a scientific outcome, not a failure.